# LangChain Expression Language

In [1]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu --upgrade

  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_openai-1.1.11-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
  Using cached openai-2.28.0-py3-none-any.whl.metadata (29 kB)
Using cached langchain-1.2.12-py3-none-any.whl (112 kB)
Using cached langchain_core-1.2.19-py3-none-any.whl (503 kB)
Using cached langchain_openai-1.1.11-py3-none-any.whl (87 kB)
Using cached openai-2.28.0-py3-none-any.whl (1.1 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1
  Attempting uninstall: langsmith━━━━━━━━━━━━━━━ 0/6 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/6 [openai]    WARNING: Ignoring invalid distribution ~angchain (/Users/james

In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [1]:
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)

---

## Accessing Previous Values using RunnablePassThrough

A runnable to passthrough inputs unchanged or with additional keys.

This runnable behaves almost like the identity function, except that it can be configured to add additional keys to the output, if the input is a dict.

The examples below demonstrate this runnable works using a few simple chains. The chains rely on simple lambdas to make the examples easy to execute and experiment with.

In [2]:
runnable = RunnableParallel(
    origin=RunnablePassthrough(),
    modified=lambda x: x+1
)

print(runnable.invoke(1)) # {'origin': 1, 'modified': 2}

def fake_llm(prompt: str) -> str: # Fake LLM for the example
    return prompt + " world"

chain = RunnableLambda(fake_llm) | {
    'original': RunnablePassthrough(), # Original LLM output
    'parsed': lambda text: text[::-1] # Parsing logic
}

chain.invoke('hello') 

{'origin': 1, 'modified': 2}


{'original': 'hello world', 'parsed': 'dlrow olleh'}

---

## Prompt + Model

In [3]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

chat = ChatOpenAI()
prompt = ChatPromptTemplate.from_template('Tell me a joke about {topic}')

chain = prompt | chat
print(chain)

first=ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}'), additional_kwargs={})]) middle=[] last=ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x10bfbd670>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10c9162d0>, root_client=<openai.OpenAI object at 0x10846f6e0>, r

In [4]:
print("first", chain.first)
print("last", chain.last)

first input_variables=['topic'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Tell me a joke about {topic}'), additional_kwargs={})]
last profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x10bfbd670> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10c9162d0> root_client=<openai.OpenAI object at 0x10846f6e0> root_async_client=<openai.AsyncOpenAI object at 0

In [5]:
# Stream:
print('\n\nStream:\n')
for s in chain.stream({"topic": "bears"}):
    print(s.content, end="", flush=True)

# Invoke:
print('\n\nInvoke:\n')
print(chain.invoke({"topic": "bears"}).content)

# Batch:
print('\n\nBatch:\n')
print(chain.batch([{"topic": "bears"}, {"topic": "bears"}, {"topic": "bears"}]))



Stream:

What do you call a bear with no teeth?

A gummy bear!

Invoke:

Why did the bear join the band? 

Because he had the right "beary" good rhythm!


Batch:

[AIMessage(content='Why did the bear dissolve in water?\n\nBecause it was polar!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 13, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DKQ2QFMAt7np7Qmbef7U9ip5mfukZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cfc46-c623-7dc2-8867-65f5ecfaf651-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 13, 'total_tokens': 26, 'input_token_details': {'audi

---

## Retrieval Augmented Generation (RAG) in LCEL

In [6]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores.faiss import FAISS

In [7]:
vectorstore = FAISS.from_texts(
    ["James Phoenix works as a data engineering and LLM consultant at JustUnderstandingData", "James has an age of 33 years old."], embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI()

In [8]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# It's the same as this, but the tuple allows for line breaks:
# {"context": retriever, "question": RunnablePassthrough()} | prompt | model | StrOutputParser()

In [9]:
chain.invoke("What company does James phoenix work at?")

'James Phoenix works at JustUnderstandingData.'

---

## Understanding How `itemgetter` Works with Piping

In [10]:
test = {
    "data": ['This is a test', 'Another entry...']
}

print(itemgetter(test))
print(itemgetter('data')(test))

operator.itemgetter({'data': ['This is a test', 'Another entry...']})
['This is a test', 'Another entry...']


### How does it work within the context of LCEL?

In [11]:
prompt = ChatPromptTemplate.from_template('''Tell me about {name}. They are {age} years old and their profession is {profession}.''')

first_chain = RunnableParallel(
    name=lambda x: "James Phoenix",
    age=lambda x: 31
)

second_chain = {
    # itemgetter is used to get the value from the dictionary from the previous step: (note this is only the previous step, not the whole chain)
    'name': itemgetter('name'),
    'age': itemgetter('age'),
    # You can not use string values, either use itemgetter or a lambda, or RunnablePassthrough
    'profession': lambda x: "Data Engineer"
}

chain = first_chain | second_chain | prompt | ChatOpenAI() | StrOutputParser()
result = chain.invoke({})
print(result)

James Phoenix is a 31-year-old data engineer who specializes in collecting, organizing, and analyzing large sets of data to help organizations make informed decisions. With a background in computer science and data analysis, James is skilled in programming languages such as Python, SQL, and Java, as well as database management systems like MySQL and MongoDB.

In his role as a data engineer, James is responsible for designing, implementing, and maintaining data pipelines, creating data models, and developing algorithms to extract valuable insights from raw data. He also collaborates with data scientists, business analysts, and other stakeholders to understand their data needs and requirements.

Outside of work, James is passionate about staying up to date on the latest developments in data engineering and technology. He enjoys attending tech conferences, participating in online forums, and contributing to open-source projects. In his free time, he also enjoys hiking, cooking, and playin